# ADA — Plug-and-Play GPU Architecture Smoke Test

**Run this notebook in Google Colab with a GPU runtime.** It clones the current ADA `main` branch, installs the repository's neural extra, and runs the committed GPU validation script. No manual file paths or code edits are required.

### What this verifies
- CUDA is visible and PyTorch can allocate on the GPU.
- The configured hybrid stack actually instantiates alternating Mamba-2 reference mixers and causal attention layers.
- The full ADA causal LM runs forward and backward with finite logits/losses.
- Critical gradient paths reach Mamba-2, attention, MoE routing, Energy Memory and the language-model head.
- AdamW changes Mamba-2 parameters and inference runs after the update.
- A JSON report and checkpoint bundle are saved under `artifacts/ada_gpu_smoke_test/`.

**Important:** synthetic tokens are used only to validate execution mechanics. A PASS is not evidence of convergence, useful learned behavior, model quality, or production GPU performance. The Mamba-2 implementation tested here is ADA's correctness-oriented PyTorch SSD reference, not a fused CUDA/Triton kernel.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU not enabled. In Colab choose Runtime → Change runtime type → T4 GPU (or another available GPU), then Runtime → Run all.')
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
from pathlib import Path
import shutil, subprocess

repo_dir = Path('/content/ADA')
if repo_dir.exists():
    shutil.rmtree(repo_dir)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', 'main',
    'https://github.com/vimalspaceton618-afk/ADA.git', str(repo_dir)
], check=True)
%cd /content/ADA
%pip install -q -e ".[neural]"
print('ADA repository installed from main.')

In [ ]:
!python scripts/ada_gpu_smoke_test.py --device cuda --steps 3 --batch-size 2 --sequence-length 64

In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown

report_path = Path('artifacts/ada_gpu_smoke_test/report.json')
report = json.loads(report_path.read_text())
display(Markdown(f"## ADA GPU test: **{report['status']}**"))
display(Markdown(
    f"- **GPU:** {report['gpu_name']}\n"
    f"- **Parameters:** {report['parameters_total']:,}\n"
    f"- **Mixer layout:** {', '.join(report['mixer_layout'])}\n"
    f"- **Final loss:** {report['last_total_loss']:.6f}\n"
    f"- **Training-step throughput:** {report['tokens_per_second_including_backward']:.2f} synthetic tokens/s\n"
    f"- **Peak GPU memory:** {report['peak_cuda_memory_mb']} MiB\n"
    f"- **Bundle:** `artifacts/ada_gpu_smoke_test/ada_gpu_smoke_test_bundle.zip`"
))
display(report)